In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

def gerar_painel_visualizacao(diretorio_dados):
    print("=== GERANDO PAINEL DE VISUALIZAÇÃO INTERATIVO ===")
    
    df_pesos = pd.read_csv(os.path.join(diretorio_dados, "historico_alocacao_HMM_EWMA_maximo_sharpe.csv"), index_col=0, parse_dates=True)
    df_ret = pd.read_csv(os.path.join(diretorio_dados, "matriz_retornos_simples_v2.csv"), index_col='Data', parse_dates=True)
    
    # Carregar CDI Externo
    caminho_cdi = os.path.join(diretorio_dados, "CDI_2010_2026.xlsx")
    df_cdi = pd.read_excel(caminho_cdi)
    df_cdi = df_cdi.rename(columns={'Date': 'Data', 'valor': 'CDI'}).set_index('Data')
    df_cdi.index = pd.to_datetime(df_cdi.index)
    
    data_ini = df_pesos.index[0]
    df_ret = df_ret.loc[data_ini:]
    
    pesos_diarios = df_pesos.reindex(df_ret.index).ffill().bfill()
    ret_ativos = df_ret[df_pesos.columns]
    ret_est = (ret_ativos * pesos_diarios).sum(axis=1)
    
    curva_est = (1 + ret_est).cumprod()
    curva_ibov = (1 + df_ret['IBOV']).cumprod()
    
    # CDI Alinhado
    cdi_alinhado = df_cdi['CDI'].reindex(df_ret.index).ffill()
    curva_cdi = (1 + cdi_alinhado).cumprod()

    top_10_ativos = df_pesos.mean().sort_values(ascending=False).head(10).index.tolist()
    df_pesos_plot = df_pesos[top_10_ativos].copy()
    df_pesos_plot['Outros'] = df_pesos.drop(columns=top_10_ativos).sum(axis=1)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        subplot_titles=("Tendência de Crescimento do Capital (Equity Curve)", 
                        "Tendência de Alocação da Carteira (Composição Mensal)")
    )

    fig.add_trace(go.Scatter(x=curva_est.index, y=curva_est, name="Estratégia HMM", line=dict(color='blue', width=3)), row=1, col=1)
    fig.add_trace(go.Scatter(x=curva_ibov.index, y=curva_ibov, name="IBOVESPA", line=dict(color='orange', width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=curva_cdi.index, y=curva_cdi, name="CDI", line=dict(color='green', dash='dash')), row=1, col=1)

    for coluna in df_pesos_plot.columns:
        fig.add_trace(
            go.Scatter(x=df_pesos_plot.index, y=df_pesos_plot[coluna], name=coluna, stackgroup='one', mode='none'),
            row=2, col=1
        )

    fig.update_layout(
        height=900,
        title_text="Dashboard de Performance e Estratégia de Alocação (2015-2025)",
        template="plotly_white",
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.update_yaxes(title_text="Capital Acumulado (Base 1)", row=1, col=1)
    fig.update_yaxes(title_text="Peso na Carteira (%)", tickformat=".0%", row=2, col=1)

    fig.show()

pasta_trabalho = r"C:\VSCodeWorkspace\TCC_Escrito\data"
gerar_painel_visualizacao(pasta_trabalho)

=== GERANDO PAINEL DE VISUALIZAÇÃO INTERATIVO ===
